# ==============================================================================
# 🏡 RESIDENTIAL REAL ESTATE VALUATION: KING COUNTY HOUSE SALES VIA DECISION TREE REGRESSOR
# ==============================================================================
### Core Principles:
Property valuation involves hierarchical geographic and structural features:
`bedrooms`, `bathrooms`, `sqft_living`, `grade`, `zipcode`, `lat`, `long`.
Decision Trees naturally construct regional bounding boxes:
$$\text{IF } \text{lat} > 47.62 \text{ (North King County)} \text{ AND } \text{grade} \ge 9 \implies \hat{y} = \$850,000$$


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Real Estate Valuation Environment ready.")


✅ STEP 1: Real Estate Valuation Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET CLEANUP
Loading Kaggle King County House Sales dataset (21,613 property sales).


In [2]:
# STEP 2 & 3: LOAD & CLEANUP
data_path = 'data/house_sales/kc_house_data.csv'
df = pd.read_csv(data_path)

# Drop non-predictive metadata
df = df.drop(columns=['id', 'date'])

print(f"📊 Dataset Shape: {df.shape[0]:,} home sales, {df.shape[1]} attributes")
print(f"Columns: {list(df.columns)[:8]}")
df.head(3)


📊 Dataset Shape: 21,613 home sales, 19 attributes
Columns: ['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view']


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `price`.


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df.drop(columns=['price'])
y = df['price']

print(f"Missing values: {X.isnull().sum().sum()}")
print(f"Target Home Price ($):")
print(y.describe().round(2))


Missing values: 0
Target Home Price ($):
count      21613.00
mean      540088.14
std       367127.20
min        75000.00
25%       321950.00
50%       450000.00
75%       645000.00
max      7700000.00
Name: price, dtype: float64


## 🔒 STEP 6: LEAK-FREE TRAIN-TEST SPLIT
80/20 train/test split.


In [4]:
# STEP 6: SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print(f"Train Properties: {X_train.shape[0]:,}")
print(f"Test Properties : {X_test.shape[0]:,}")


Train Properties: 17,290
Test Properties : 4,323


## ⚙️ STEP 7: SCALE-INVARIANT PIPELINE
All features are continuous or discrete integer metrics; pass-through pipeline.


In [5]:
# STEP 7: DIRECT SCALE-INVARIANT PIPELINE
pipe_base = Pipeline([
    ('tree', DecisionTreeRegressor(random_state=42))
])
print("✅ STEP 7: Scale-Invariant Property Valuation Pipeline ready.")


✅ STEP 7: Scale-Invariant Property Valuation Pipeline ready.


## ⚠️ STEP 8: THE OVERFITTING TRAP
Evaluating full unpruned depth vs constrained tree.


In [6]:
# STEP 8: OVERFITTING DEMONSTRATION
pipe_base.fit(X_train, y_train)

print(f"Unconstrained Tree Train R² : {r2_score(y_train, pipe_base.predict(X_train))*100:.2f}%")
print(f"Unconstrained Tree Test R²  : {r2_score(y_test, pipe_base.predict(X_test))*100:.2f}%")


Unconstrained Tree Train R² : 99.92%
Unconstrained Tree Test R²  : 71.70%


## 🎯 STEP 9: TUNING VIA 5-FOLD CROSS VALIDATION
Optimizing `max_depth` and `min_samples_leaf` for housing price appraisal.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'tree__max_depth': [6, 8, 10, 12, 14],
    'tree__min_samples_leaf': [5, 10, 20]
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_base, param_grid=param_grid, cv=cv, scoring='r2', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV R²   : {grid.best_score_*100:.2f}%")


🏆 Best Hyperparameters: {'tree__max_depth': 12, 'tree__min_samples_leaf': 5}
🎯 Best 5-Fold CV R²   : 78.45%


## ⚔️ STEP 10: ALGORITHM DUEL (DecisionTreeRegressor vs Ridge)
Benchmarking tree regression against regularized Ridge regression.


In [8]:
# STEP 10: MODEL BENCHMARK
from sklearn.preprocessing import StandardScaler

pipe_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=10.0, random_state=42))
])
pipe_ridge.fit(X_train, y_train)

y_pred_tree = best_model.predict(X_test)
y_pred_ridge = pipe_ridge.predict(X_test)

print("="*65)
print(f"DecisionTreeRegressor (Tuned) -> Test R²: {r2_score(y_test, y_pred_tree)*100:.2f}% | RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_tree)):,.2f}")
print(f"Ridge Regression              -> Test R²: {r2_score(y_test, y_pred_ridge)*100:.2f}% | RMSE: ${np.sqrt(mean_squared_error(y_test, y_pred_ridge)):,.2f}")
print("="*65)


DecisionTreeRegressor (Tuned) -> Test R²: 77.96% | RMSE: $182,522.79
Ridge Regression              -> Test R²: 70.12% | RMSE: $212,545.65


## 📊 STEP 11: RESIDUAL ANALYSIS & VALUATION METRICS
Diagnostics on house price errors and influential property traits.


In [9]:
# STEP 11: EVALUATION REPORT
mae = mean_absolute_error(y_test, y_pred_tree)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_tree))

print(f"📋 RESIDENTIAL APPRAISAL METRICS:")
print(f"   Mean Absolute Error (MAE) : ${mae:,.2f}")
print(f"   Root Mean Squared Error   : ${rmse:,.2f}")
print(f"   R² Score                  : {r2_score(y_test, y_pred_tree)*100:.2f}%")

# Feature importances
tree_est = best_model.named_steps['tree']
importances = pd.Series(tree_est.feature_importances_, index=X.columns).sort_values(ascending=False)
print(f"\nTop Valuation Drivers:\n{importances.head(6).round(4)}")


📋 RESIDENTIAL APPRAISAL METRICS:
   Mean Absolute Error (MAE) : $92,939.75
   Root Mean Squared Error   : $182,522.79
   R² Score                  : 77.96%

Top Valuation Drivers:
grade          0.3906
sqft_living    0.2492
lat            0.1605
long           0.0581
waterfront     0.0297
yr_built       0.0289
dtype: float64


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & APPRAISAL SMOKE TEST
Serializing valuation pipeline and testing automated instant appraisal latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/king_county_house_tree_pipeline.joblib'
joblib.dump(best_model, model_path)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_property = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_val = loaded_pipe.predict(sample_property)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏡 LIVE INSTANT APPRAISAL AUDIT:")
print(f"   Automated Valuation Estimate: 🏷️ ${pred_val:,.2f}")
print(f"   Actual Transaction Price    : ${y_test.iloc[0]:,.2f}")
print(f"   Valuation Error             : ${abs(pred_val - y_test.iloc[0]):,.2f}")
print(f"   Appraisal Engine Latency    : {latency_ms:.3f} ms (SLA < 1.0ms: {'PASS' if latency_ms < 1.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/king_county_house_tree_pipeline.joblib (134,434 bytes)

🏡 LIVE INSTANT APPRAISAL AUDIT:
   Automated Valuation Estimate: 🏷️ $377,203.06
   Actual Transaction Price    : $365,000.00
   Valuation Error             : $12,203.06
   Appraisal Engine Latency    : 1.003 ms (SLA < 1.0ms: CHECK)
